In [11]:
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from utils import RAW_DATA_DIR
import pandas as pd

In [12]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    preprocessed_text = ' '.join(tokens)
    return preprocessed_text

In [13]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, lowercase=True, remove_punctuation=True,
                 remove_numbers=True, remove_stopwords=True,
                 stemming=True, language='english'):
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation
        self.remove_numbers = remove_numbers
        self.remove_stopwords = remove_stopwords
        self.stemming = stemming
        self.language = language

        # Initialize stemmer and stopwords
        if self.stemming:
            self.stemmer = PorterStemmer()
        if self.remove_stopwords:
            self.stop_words = set(stopwords.words(self.language))

    def fit(self, X, y=None):
        # Nothing to fit for this transformer
        return self

    def transform(self, X, y=None):
        # Apply preprocessing to each document in X
        return [self._preprocess_text(text) for text in X]

    def _preprocess_text(self, text):
        if self.lowercase:
            text = text.lower()

        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', '', text)

        if self.remove_numbers:
            text = re.sub(r'\d+', '', text)

        tokens = text.split()

        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in self.stop_words]

        if self.stemming:
            tokens = [self.stemmer.stem(token) for token in tokens]

        preprocessed_text = ' '.join(tokens)
        return preprocessed_text

In [14]:
# lowercase doesn't matter when stemming=True
text_pr_pipeline = Pipeline([
    ('preprocessor', TextPreprocessor(lowercase=False, remove_punctuation=True,
                 remove_numbers=True, remove_stopwords=True,
                 stemming=True)),
])

texts = ["This is a sample text with some numbers 123 and punctuation!", "Another example..."]
processed_texts = text_pr_pipeline.fit_transform(texts)
processed_texts

['thi sampl text number punctuat', 'anoth exampl']

In [15]:
df = pd.read_csv(RAW_DATA_DIR / "spam_assassin.csv")
df.head()

,id,mail,category,spam
0,00001.1a31cc283af0060967a233d26548a6ce,Return-Path: <exmh-workers-admin@spamassassin....,easy_ham_2,0
1,00001.317e78fa8ee2f54cd4890fdc09ba8176,From ilug-admin@linux.ie Tue Aug 6 11:51:02 ...,spam_2,1
2,00001.7848dde101aa985090474a91ec93fcf0,From 12a1mailbot1@web.de Thu Aug 22 13:17:22 ...,spam,1
3,00001.7c53336b37003a9286aba55d2945844c,From exmh-workers-admin@redhat.com Thu Aug 22...,easy_ham,0
4,00001.7c7d6921e671bbe18ebb5f893cd9bb35,Return-Path: Fool@motleyfool.com\nDelivery-Dat...,hard_ham,0


In [18]:
# cross validation needs to be outside the pipeline
text_pipeline = Pipeline([
    ('preprocessor', TextPreprocessor(lowercase=False, remove_punctuation=True,
                 remove_numbers=True, remove_stopwords=True,
                 stemming=True)),
    ('vectorizer', CountVectorizer()),
    ('classifier', RandomForestClassifier(class_weight="balanced"))
])

param_grid = {
    'classifier__n_estimators': [50, 100, 250, 500],
    'classifier__max_depth': [5, 10, 25, 50]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(text_pipeline, param_grid=param_grid, cv=cv, scoring='accuracy')

grid_search.fit(df.mail, df.spam)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

Best parameters: {'classifier__max_depth': 50, 'classifier__n_estimators': 100}
Best cross-validation accuracy: 0.9804591867527721
